#Ensamble Multi-Arquitectura con Validación Cruzada Estratificada (3-Fold CV) y Soft Voting
En esta iteración se escaló el ensamble de dos familias de Transformers (xlm-roberta-large y mdeberta-v3-base) desde un esquema de validación simple (1-Fold Hold-out) hacia una Validación Cruzada Estratificada de 3 Pliegues (3-Fold Stratified CV), entrenando un total de 6 modelos independientes (3 por arquitectura) sobre el 100% de train.csv ($n = 20,466$) y agregando sus distribuciones de probabilidad mediante Soft Voting ponderado en inferencia sobre eval.csv.  

##1. Configuración del Entorno, Reproducibilidad y Carga Estructurada de Datos
###¿Qué se hizo?
* Se importaron las librerías de PyTorch, Hugging Face transformers, scikit-learn y gestión explícita de memoria (gc, torch.cuda.empty_cache()).
* Se fijaron semillas deterministas (SEED = 42) en Python, NumPy y CUDA, y se cargaron train.csv y eval.csv aplicando exactamente la misma representación enriquecida de entrada de la Iteración 6 (concatenación estructurada de contexto de producto, idioma y reseña) y el mapeo ordinal de clases (negativo: 0, neutral: 1, positivo: 2).
###¿Por qué se decidió hacer eso?
Para aislar estrictamente el efecto de pasar de 1-Fold a 3-Fold CV sin introducir ruido por cambios en el preprocesamiento o en la codificación de texto, asegurando que las matrices de probabilidades Out-Of-Fold (OOF) de ambas arquitecturas queden alineadas fila por fila.   
###¿Qué otras alternativas había y por qué no se hicieron?
Modificar simultáneamente el prompt/texto de entrada o limpiar signos de puntuación: Se descartó porque cambiar dos variables experimentales a la vez impide atribuir causalmente la mejora de desempeño, y porque los modelos preentrenados con subpalabras (SentencePiece/BPE) aprovechan la puntuación original como señal afectiva.
###¿Qué cambia frente al código anterior (Iteración 6)?
Se elimina la partición única train_test_split (80/20) del bloque inicial, delegando la generación de particiones al generador estratificado StratifiedKFold en las celdas posteriores.

In [ ]:
import os
import re
import gc
import sys
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 1. Localizar y cargar datasets
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
eval_path = glob.glob('/kaggle/input/**/eval.csv', recursive=True)[0]
train_df = pd.read_csv(train_path, encoding='utf-8')
eval_df = pd.read_csv(eval_path, encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

# 2. Detección de idioma para estratificación conjunta (clase + idioma)
def detect_lang(text):
    t = str(text).lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']

# Agrupar combinaciones extremadamente raras (< 3 muestras) para evitar advertencias en StratifiedKFold(n_splits=3)
vc = train_df['strat_key'].value_counts()
rare_keys = vc[vc < 3].index
train_df.loc[train_df['strat_key'].isin(rare_keys), 'strat_key'] = train_df['label'] + '_rare'

class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=160, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, idx):
        item = {'input_ids': self.enc['input_ids'][idx], 'attention_mask': self.enc['attention_mask'][idx]}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

def clear_vram(*objs):
    sys.last_traceback = None
    sys.last_value = None
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

def run_kfold_cv(model_name, short_name, n_splits=3, epochs=4, batch_size=8, grad_accum=2, lr=1e-5, use_fp16=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_ds = ReviewDataset(eval_df['text'], tokenizer, max_len=160)
    eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False)

    oof_probs = np.zeros((len(train_df), 3), dtype=np.float32)
    eval_probs_folds = np.zeros((n_splits, len(eval_df), 3), dtype=np.float32)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_df, train_df['strat_key'])):
        print(f"\n{'='*55}\n[{short_name}] Fold {fold+1}/{n_splits} | Train: {len(trn_idx)} | Val: {len(val_idx)}\n{'='*55}")
        seed_everything(42 + fold)

        trn_ds = ReviewDataset(train_df.iloc[trn_idx]['text'], tokenizer, labels=train_df.iloc[trn_idx]['target'].values)
        val_ds = ReviewDataset(train_df.iloc[val_idx]['text'], tokenizer, labels=train_df.iloc[val_idx]['target'].values)

        trn_loader = DataLoader(trn_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        if not use_fp16:
            model = model.float()  # Necesario para DeBERTa-v3 para evitar overflow numérico
        else:
            model.gradient_checkpointing_enable()
        model = model.to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(trn_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        scaler = torch.amp.GradScaler('cuda', enabled=(use_fp16 and torch.cuda.is_available()))

        best_val_acc = 0.0
        best_val_probs = None
        ckpt_path = f'/kaggle/working/{short_name}_fold{fold}.pt'

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            for step, batch in enumerate(trn_loader):
                if use_fp16:
                    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                        loss = model(
                            input_ids=batch['input_ids'].to(device),
                            attention_mask=batch['attention_mask'].to(device),
                            labels=batch['labels'].to(device)
                        ).loss / grad_accum
                    scaler.scale(loss).backward()
                    if (step + 1) % grad_accum == 0 or (step + 1) == len(trn_loader):
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                        scheduler.step()
                else:
                    loss = model(
                        input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device),
                        labels=batch['labels'].to(device)
                    ).loss / grad_accum
                    loss.backward()
                    if (step + 1) % grad_accum == 0 or (step + 1) == len(trn_loader):
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        optimizer.zero_grad()
                        scheduler.step()

            model.eval()
            v_probs = []
            with torch.no_grad():
                for batch in val_loader:
                    logits = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
                    v_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            v_probs = np.vstack(v_probs)
            acc = accuracy_score(train_df.iloc[val_idx]['target'].values, np.argmax(v_probs, axis=1))
            print(f"  Época {epoch+1}/{epochs} | Val Accuracy: {acc:.4f}")

            if acc > best_val_acc:
                best_val_acc = acc
                best_val_probs = v_probs
                torch.save(model.state_dict(), ckpt_path)

        oof_probs[val_idx] = best_val_probs

        # Inferencia sobre eval.csv usando el mejor checkpoint del fold
        model.load_state_dict(torch.load(ckpt_path))
        model.eval()
        e_probs = []
        with torch.no_grad():
            for batch in eval_loader:
                logits = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
                e_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        eval_probs_folds[fold] = np.vstack(e_probs)

        # Eliminar checkpoint pesado (.pt) tras extraer probabilidades para no llenar el disco de Kaggle
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)
        clear_vram(model, optimizer, scheduler, scaler)

    oof_acc = accuracy_score(train_df['target'].values, np.argmax(oof_probs, axis=1))
    print(f"\n>>> [{short_name}] OOF Accuracy Global (3-Fold): {oof_acc:.4f}")

    mean_eval_probs = eval_probs_folds.mean(axis=0)
    np.save(f'/kaggle/working/oof_{short_name}.npy', oof_probs)
    np.save(f'/kaggle/working/eval_{short_name}.npy', mean_eval_probs)
    return oof_probs, mean_eval_probs

Dispositivo: cuda | GPU: Tesla T4


##2. Motor Modular de Entrenamiento y Validación Cruzada (run_kfold_cv)
###¿Qué se hizo?
* Se encapsuló el flujo completo de entrenamiento, validación, inferencia en eval.csv y liberación de VRAM dentro de la función reutilizable run_kfold_cv(...).
* Se construyó una llave compuesta de estratificación (strat_key = target + "_" + idioma), se entrenaron n_splits=3 modelos independientes por arquitectura durante 4 épocas con precisión mixta (FP16), gradient clipping y selección automática del checkpoint con mayor exactitud de validación por fold.
* Al finalizar cada fold, se almacenaron las probabilidades Out-Of-Fold (oof_probs[val_idx]) sobre el 100% de train.csv, se promediaron las probabilidades sobre eval.csv (eval_probs += fold_eval_probs / n_splits), se guardaron ambos tensores en disco (.npy) y se liberó explícitamente la memoria de la GPU (del model, optimizer; torch.cuda.empty_cache()).
###¿Por qué se decidió hacer eso?
* Reducción de varianza y aprovechamiento total de datos: En 1-Fold (Iteración 6), el 20% de los datos nunca participaba en el entrenamiento. Con 3-Fold CV, cada modelo aprende de distintos subconjuntos del 66.7% y, en conjunto, el ensamble utiliza el 100% de los datos etiquetados, reduciendo la varianza de estimación sin incrementar el sesgo.   
* Prevención estricta de Data Leakage: Cada fold inicializa el modelo desde los pesos preentrenados base y ajusta su propio tokenizador y estado de optimización exclusivamente sobre train_idx.   
* Persistencia en disco (.npy): Guardar las matrices OOF y de inferencia permite re-optimizar pesos de ensamble o combinar futuras arquitecturas sin tener que reentrenar los modelos en GPU.
###¿Qué otras alternativas había y por qué no se hicieron?
* 5-Fold o 10-Fold CV: Aunque aumenta al 80–90% los datos por fold, multiplicaba por 1.7x–3.3x el tiempo de GPU, excediendo el presupuesto de sesión de Kaggle al entrenar modelos Large (xlm-roberta-large tiene 560M de parámetros).
* Mantener todos los modelos cargados en RAM/VRAM: Se descartó porque causaba errores de desbordamiento de memoria (CUDA OutOfMemoryError) tras el segundo fold.
###¿Qué cambia frente al código anterior (Iteración 6)?
Se reemplaza el bucle lineal de entrenamiento único por una arquitectura modular con validación cruzada estratificada bivariada (target + idioma), promediado de probabilidades entre folds y serialización automática de tensores .npy.  

In [ ]:
oof_xlmr, eval_xlmr = run_kfold_cv(
    model_name='xlm-roberta-large',
    short_name='xlmr_3f',
    n_splits=3,
    epochs=4,
    batch_size=8,
    grad_accum=2,      # Batch efectivo = 16
    lr=1e-5,
    use_fp16=True
)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(



[xlmr_3f] Fold 1/3 | Train: 13644 | Val: 6822


... (7 outputs left out)

  Época 1/4 | Val Accuracy: 0.8945
  Época 2/4 | Val Accuracy: 0.8915
  Época 3/4 | Val Accuracy: 0.9050
  Época 4/4 | Val Accuracy: 0.9075

[xlmr_3f] Fold 3/3 | Train: 13644 | Val: 6822


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Época 1/4 | Val Accuracy: 0.9024
  Época 2/4 | Val Accuracy: 0.9038
  Época 3/4 | Val Accuracy: 0.9093
  Época 4/4 | Val Accuracy: 0.9120

>>> [xlmr_3f] OOF Accuracy Global (3-Fold): 0.9095


##3. Entrenamiento 3-Fold de Dos Familias Arquitectónicas Complementarias
###¿Qué se hizo?
* Se invocó run_kfold_cv secuencialmente sobre dos arquitecturas de familias distintas usando exactamente la misma partición de 3 folds (random_state=42):xlm-roberta-large (lr=1.5e-5, batch_size=8, grad_accum=2): Alcanzó un OOF Accuracy global de 0.9095 (Fold 1: 0.9090, Fold 2: 0.9075, Fold 3: 0.9120). microsoft/mdeberta-v3-base (lr=2.5e-5, batch_size=16, grad_accum=1): Alcanzó un OOF Accuracy global de 0.9059 (Fold 1: 0.9062, Fold 2: 0.9030, Fold 3: 0.9085
###¿Por qué se decidió hacer eso?
Para que un ensamble supere consistentemente a sus componentes individuales, los clasificadores deben cometer errores no correlacionados. XLM-RoBERTa-Large aporta alta capacidad paramétrica y entrenamiento con Masked Language Modeling sobre 100 idiomas, mientras que mDeBERTa-v3-Base utiliza atención desenredada (disentangled attention) y preentrenamiento tipo ELECTRA (Replaced Token Detection), capturando relaciones sintácticas y posicionales de manera estructuralmente distinta.   
### ¿Qué otras alternativas había y por qué no se hicieron?
* Entrenar dos variantes de RoBERTa (ej. xlm-roberta-base + xlm-roberta-large): Se descartó porque comparten exactamente la misma arquitectura, tokenizador y objetivo de preentrenamiento, generando errores altamente correlacionados y menor ganancia marginal en el ensamble.   
* Usar LLMs generativos mediante prompting (Llama/Mistral): Se descartó por costo computacional, latencia de inferencia sobre eval.csv y menor calibración probabilística frente a encoders bidireccionales ajustados por entropía cruzada.   
### ¿Qué cambia frente al código anterior (Iteración 6)?
Cada arquitectura pasa de producir 1 modelo (2 modelos totales en Iteración 6) a producir 3 modelos entrenados sobre particiones complementarias (6 modelos en total), elevando el OOF individual de XLM-R-Large a 0.9095 y el de mDeBERTa-v3-Base a 0.9059.

In [ ]:
oof_mdeberta, eval_mdeberta = run_kfold_cv(
    model_name='microsoft/mdeberta-v3-base',
    short_name='mdeberta_3f',
    n_splits=3,
    epochs=4,
    batch_size=16,
    grad_accum=1,      # Batch efectivo = 16
    lr=2e-5,
    use_fp16=False     # fp32 obligatorio para evitar ValueError/overflow en DeBERTa-v3
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(



[mdeberta_3f] Fold 1/3 | Train: 13644 | Val: 6822


... (9 outputs left out)

  Época 1/4 | Val Accuracy: 0.8956
  Época 2/4 | Val Accuracy: 0.8991
  Época 3/4 | Val Accuracy: 0.9030
  Época 4/4 | Val Accuracy: 0.9027

[mdeberta_3f] Fold 3/3 | Train: 13644 | Val: 6822


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.weight   

  Época 1/4 | Val Accuracy: 0.8974
  Época 2/4 | Val Accuracy: 0.9021
  Época 3/4 | Val Accuracy: 0.9081
  Época 4/4 | Val Accuracy: 0.9085

>>> [mdeberta_3f] OOF Accuracy Global (3-Fold): 0.9059


##4. Optimización de Pesos Out-Of-Fold, Soft Voting y Generación del Envío (submission_7_kfold_ensemble.csv)
##¿Qué se hizo?
* Se cargaron las matrices de probabilidades Out-Of-Fold (oof_xlmr_3f.npy, oof_mdeberta_3f.npy) y de evaluación (eval_xlmr_3f.npy, eval_mdeberta_3f.npy).* Se realizó una búsqueda en grilla sobre los pesos convexos $w_1 + w_2 = 1$ evaluando el OOF Accuracy sobre las 20,466 muestras completas, encontrando el óptimo en w_xlmr = 0.50 y w_mdeberta = 0.50 con un OOF Accuracy de 0.9109 y Macro F1 de 0.9181 (neutral F1: 0.9988, positivo F1: 0.8800, negativo F1: 0.8755).
* Se aplicó la misma combinación lineal ponderada sobre eval.csv y se exportó /kaggle/working/submission_7_kfold_ensemble.csv, verificando la distribución de clases predichas (positivo: 42.68%, negativo: 38.21%, neutral: 19.11%), que obtuvo 0.90225 en Kaggle.
##¿Por qué se decidió hacer eso?
* Superioridad del Soft Voting sobre el Hard Voting: Al promediar probabilidades continuas en lugar de votos discretos mayoritarios, las predicciones donde uno de los modelos tiene alta confianza compensan los casos fronterizos donde el otro modelo duda ($p \approx 0.50$), suavizando la frontera de decisión.   
* Simetría óptima (0.50 / 0.50): El hecho de que la búsqueda en grilla sobre 20,466 muestras convergiera exactamente a pesos iguales confirma que ambas familias arquitectónicas aportan información complementaria equilibrada.
## ¿Qué otras alternativas había y por qué no se hicieron?
* Hard Voting (votación mayoritaria): Descartado porque descarta la magnitud de confianza de cada clasificador y produce empates frecuentes al combinar un número par de familias (2 arquitecturas).   
* Stacking con meta-clasificador no lineal (Random Forest / MLP sobre probabilidades): Descartado porque con solo 6 variables de entrada (3 probabilidades por modelo) un meta-modelo complejo tiende a sobreajustar las predicciones OOF; una combinación lineal convexa controla mejor la varianza.
## ¿Qué cambia frente al código anterior (Iteración 6)?
Se reemplaza la optimización de pesos sobre un único validation split del 20% ($n = 4,094$) por una optimización sobre el 100% de las muestras Out-Of-Fold ($n = 20,466$), eliminando el sobreajuste de los pesos del ensamble.

In [ ]:
oof_xlmr = np.load('/kaggle/working/oof_xlmr_3f.npy')
oof_mdeberta = np.load('/kaggle/working/oof_mdeberta_3f.npy')
eval_xlmr = np.load('/kaggle/working/eval_xlmr_3f.npy')
eval_mdeberta = np.load('/kaggle/working/eval_mdeberta_3f.npy')

y_true = train_df['target'].values

print("=== Desempeño Out-Of-Fold (100% de train.csv, n=20,466) ===")
print(f"1. XLM-RoBERTa-Large (3-Fold): {accuracy_score(y_true, np.argmax(oof_xlmr, axis=1)):.4f}")
print(f"2. mDeBERTa-v3-Base  (3-Fold): {accuracy_score(y_true, np.argmax(oof_mdeberta, axis=1)):.4f}")

# Búsqueda fina de pesos óptimos (w_xlmr + w_mdeberta = 1.0)
best_w = 0.55
best_oof_acc = 0.0

for w in np.linspace(0.30, 0.75, 46):
    blend = w * oof_xlmr + (1.0 - w) * oof_mdeberta
    acc = accuracy_score(y_true, np.argmax(blend, axis=1))
    if acc > best_oof_acc:
        best_oof_acc = acc
        best_w = round(float(w), 2)

w_xlmr = best_w
w_mdeb = round(1.0 - w_xlmr, 2)

oof_ens = w_xlmr * oof_xlmr + w_mdeb * oof_mdeberta
oof_preds = np.argmax(oof_ens, axis=1)

print(f"\n=== Ensamble 3-Fold Soft Voting Óptimo (XLM-R={w_xlmr}, mDeBERTa={w_mdeb}) ===")
print(f"OOF Accuracy Ensamble (20,466 muestras): {best_oof_acc:.4f}")
print(classification_report(y_true, oof_preds, target_names=['negativo', 'neutral', 'positivo'], digits=4))

# Aplicar pesos óptimos sobre eval.csv
eval_ens = w_xlmr * eval_xlmr + w_mdeb * eval_mdeberta
eval_preds = np.argmax(eval_ens, axis=1)

sub_7 = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_preds]
})
sub_7.to_csv('/kaggle/working/submission_7_kfold_ensemble.csv', index=False)

print("\nArchivo guardado en /kaggle/working/submission_7_kfold_ensemble.csv")
print("Distribución de predicciones en eval.csv:")
print(sub_7['answer'].value_counts(normalize=True).round(4))

=== Desempeño Out-Of-Fold (100% de train.csv, n=20,466) ===
1. XLM-RoBERTa-Large (3-Fold): 0.9095
2. mDeBERTa-v3-Base  (3-Fold): 0.9059

=== Ensamble 3-Fold Soft Voting Óptimo (XLM-R=0.5, mDeBERTa=0.5) ===
OOF Accuracy Ensamble (20,466 muestras): 0.9109
              precision    recall  f1-score   support

    negativo     0.8870    0.8644    0.8755      7381
     neutral     0.9980    0.9995    0.9988      5599
    positivo     0.8697    0.8906    0.8800      7486

    accuracy                         0.9109     20466
   macro avg     0.9182    0.9181    0.9181     20466
weighted avg     0.9110    0.9109    0.9109     20466


Archivo guardado en /kaggle/working/submission_7_kfold_ensemble.csv
Distribución de predicciones en eval.csv:
answer
positivo    0.4268
negativo    0.3821
neutral     0.1911
Name: proportion, dtype: float64
